# Satellite based

In [ ]:
import os, re, sys, math, json, time, random, glob
project_dir = f'/home/{os.environ["USER"]}/FireScope'
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from custom_datasets.dataset import CustomDataset, collate_vlm
from config import DATA_DIR
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

from training.train import SSIM, grad_loss, init_distributed
from models.unet import FilmUnet, Unet
from models.sam import SamFiLMTransformerDecoder#, SamNoFiLMTransformerDecoder
from models.dino import DINOv3FiLMTransformerDecoder, DINOv3NoFiLMTransformerDecoder
from models.vlm_generator import QwenVLTransformerRaster
local_rank, world_size, rank = init_distributed()

device = torch.device(f'cuda:{local_rank}' if torch.cuda.is_available() else 'cpu')

ssim = SSIM().to(device)
huber = nn.SmoothL1Loss(beta=1)
mse = nn.MSELoss()
mae = nn.L1Loss()
from typing import Dict, List, Optional
from PIL import Image as PILImage
from models.vlm_generator import QwenVLTransformerRaster
from prompts import raster_generation_prompt as main_prompt
def evaluate(
    model_cpt: str,
    satellite_image_path: str,
    raster_path: str,
    out_dir: str,
    device=device,
    ssim=ssim,
    huber=huber,
    mse=mse,
    mae=mae,
    oracle: bool = False,
    model_type=Unet,
    tensor_datatype: bool = True,           # <<< set to False for Qwen VLM path (PIL images)
    climate_json: str = "/work/wildfirerisk/climate_data.json",
    climate_cond: bool = False,
    vlm_pred_file: Optional[str] = None,
    model_kwargs: Optional[dict] = None,    # optional: pass custom ctor args here
    with_counterfactual: bool = False,
):
    """
    If model_type is QwenVLTransformerRaster and tensor_datatype=False:
      - dataset returns PIL images
      - we build text prompts and call model(images, prompts)
    Otherwise:
      - dataset returns tensors and we call model(tensor_img, cond=...)
    """

    if vlm_pred_file and not oracle:
        raise Exception("Must have oracle enabled when conditioning on vlm output!")

    # ----------------------------------------------------
    # Instantiate model
    # ----------------------------------------------------
    model_kwargs = model_kwargs or {}
    is_vlm = (model_type is QwenVLTransformerRaster)

    if is_vlm:
        # Reasonable eval defaults (match your train code shape/dtype)
        defaults = dict(
            model_id="Qwen/Qwen2.5-VL-7B-Instruct",
            out_ch=1,
            d_model=512, depth=8, heads=8, grid_hw=(43,43),
            dtype=torch.bfloat16,
            freeze_qwen=True,  # eval: freezing is fine; keeps memory lower
        )
        defaults.update(model_kwargs)
        model = model_type(**defaults)
    else:
        if climate_cond:
            cond_dim=60
        elif oracle:
            cond_dim=1
        else:
            cond_dim=None
        if cond_dim:
            model = model_type(cond_dim=cond_dim)
        else:
            model = model_type()

    # load checkpoint
    state = torch.load(model_cpt, map_location="cpu")
    model.load_state_dict(state["model_state"] if "model_state" in state else state)
    model.to(device)
    model.eval()

    # make output dir
    os.makedirs(out_dir, exist_ok=True)

    # ----------------------------------------------------
    # Dataset / Loader
    # ----------------------------------------------------
    # For Qwen VLM path, we want PIL images. For tensor models, we want tensors.
    image_return_mode = "pil" if (is_vlm and not tensor_datatype) else "tensor"
    climate_return_mode = "raw" if (is_vlm and not tensor_datatype) else "vector"
    val_ds = CustomDataset(
        satellite_image_path,
        raster_path,
        climate_dir=climate_json,
        with_cheating=oracle,
        vlm_pred_json=vlm_pred_file,
        image_return=image_return_mode,   # <<< key toggle
        climate_return=climate_return_mode,              # you build text prompts from this
        counterfactual=with_counterfactual,
    )

    val_sampler = DistributedSampler(val_ds, shuffle=False, drop_last=False) if dist.is_initialized() else None
    val_loader = DataLoader(
        val_ds,
        batch_size=32,  # VLM path is heavier
        shuffle=False,
        sampler=val_sampler,
        num_workers=0,
        drop_last=False,
        collate_fn=(collate_vlm if not tensor_datatype else None),  # <<< KEY FIX
    )

    totals = {"recon": 0.0, "ssim": 0.0, "grad": 0.0, "mse": 0.0, "mae": 0.0}
    n = 0

    with torch.no_grad():
        for batch in tqdm(val_loader):
            img, climate, rast, cheat, base = batch

            if not tensor_datatype:
                # img is List[PIL.Image], build prompts and call the model
                prompts = [main_prompt.build_prompt(c) for c in climate]
                pred = model(img, prompts)   # Qwen wrapper handles PIL→processor
            else:
                img = img.to(device, non_blocking=True)
                if climate_cond:
                    climate = climate.to(device, non_blocking=True)
                    pred = model(img, cond=climate)
                elif oracle or vlm_pred_file:
                    cheat = cheat.to(device, non_blocking=True)
                    pred = model(img, cond=cheat)
                else:
                    pred = model(img)

            rast = rast.to(device, non_blocking=True)

            l_recon = huber(pred, rast)
            l_mse   = mse(pred, rast)
            l_mae   = mae(pred, rast)

            pred_01 = (pred * 0.5 + 0.5).clamp(0, 1)
            rast_01 = (rast * 0.5 + 0.5).clamp(0, 1)

            l_ssim = 1.0 - ssim(pred_01, rast_01)
            l_grad = grad_loss(pred, rast)

            bs = rast.size(0)
            totals["recon"] += l_recon.item() * bs
            totals["mse"]   += l_mse.item()   * bs
            totals["mae"]   += l_mae.item()   * bs
            totals["ssim"]  += l_ssim.item()  * bs
            totals["grad"]  += l_grad.item()  * bs
            n += bs

            # --- Optional: save predicted rasters (as .npy) ---
            # (kept same behavior you had, with filename fix)
            preds_np = pred_01.detach().cpu().numpy()  # [B,1,H,W]
            for i in range(bs):
                single_pred = preds_np[i, 0] if preds_np.ndim == 4 else preds_np[i]
                # base may be list of names or a single string
                filename = os.path.basename(base[i]) if isinstance(base, (list, tuple)) else os.path.basename(base)
                name = filename[:-4] if filename.lower().endswith(".npy") else filename

                out_path = os.path.join(out_dir, f"{name}.npy")
                np.save(out_path, single_pred)

    # --- Distributed average of ALL metrics ---
    if dist.is_available() and dist.is_initialized():
        t = torch.tensor(
            [totals["recon"], totals["ssim"], totals["grad"], totals["mse"], totals["mae"], float(n)],
            device=device
        )
        dist.all_reduce(t, op=dist.ReduceOp.SUM)
        totals["recon"], totals["ssim"], totals["grad"], totals["mse"], totals["mae"], n = t.tolist()

    return {k: v / max(n, 1) for k, v in totals.items()}


In [ ]:
eval_folders = [
{
    "satellite_image_path": f"{DATA_DIR}/europe_dataset/no_fire_images",
    "raster_path": f"{DATA_DIR}/europe_dataset/no_fire_masks",
    "out_dir": "europe_negatives",
},
{
    'satellite_image_path': f"{DATA_DIR}/large_dataset/satellite_images_full/test",
    'raster_path': f"{DATA_DIR}/large_dataset/normalised_risk_rasters/test",
    'out_dir': 'testset',
}] + [
{
    "satellite_image_path": f"/work/wildfirerisk/europe_dataset/fire_images/{yr-1}",
    "raster_path": f"/work/wildfirerisk/europe_dataset/fire_masks/{yr}",
    "out_dir": f"europe_fires/{yr}"}
    for yr in range(2018, 2026)]

eval_models_and_params = [
    {
        "model_cpt": f"{DATA_DIR}/trainings/unet_baseline/ckpts/fusion_epoch556.pt",
        "out_dir": f"{DATA_DIR}/evaluations/unet_baseline",
        "model_type": Unet,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/unet_cond_climate/ckpts/fusion_epoch252.pt",
        "out_dir": f"{DATA_DIR}/evaluations/unet_cond_climate",
        "climate_cond": True,
        "model_type": FilmUnet,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/unet_cond_cheating/ckpts/fusion_epoch157.pt",
        "out_dir": f"{DATA_DIR}/evaluations/unet_oracle_vlm_groundtruth_no_cot",
        "vlm_pred_file": "/work/wildfirerisk/vlm_forward_pass_predictions.json",
        "oracle": True,
        "model_type": FilmUnet,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/unet_cond_cheating/ckpts/fusion_epoch157.pt",
        "out_dir": f"{DATA_DIR}/evaluations/unet_oracle_vlm_groundruth",
        "vlm_pred_file": "/work/wildfirerisk/vlm_cot_predictions.json",
        "oracle": True,
        "model_type": FilmUnet,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/unet_baseline/ckpts/fusion_epoch345.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/unet_baseline",
        "model_type": Unet,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/unet_cond_oracle/ckpts/fusion_epoch195.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/unet_oracle_vlm",
        "vlm_pred_file": "/work/wildfirerisk/vlm_cot_predictions.json",
        "oracle": True,
        "model_type": FilmUnet,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/unet_cond_oracle_no_cot/ckpts/fusion_epoch095.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/unet_oracle_vlm_no_cot",
        "vlm_pred_file": "/work/wildfirerisk/vlm_forward_pass_predictions.json",
        "oracle": True,
        "model_type": FilmUnet,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/unet_cond_climate/ckpts/fusion_epoch045.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/unet_cond_climate",
        "climate_cond": True,
        "model_type": FilmUnet,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/sam/ckpts/fusion_epoch094.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/sam",
        "climate_cond": True,
        "model_type": SamFiLMTransformerDecoder,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/vlm_generator/ckpts/best.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/vlm_generator",
        "climate_cond": True,
        "model_type": QwenVLTransformerRaster,
        "tensor_datatype": False
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/unet_cond_oracle/ckpts/fusion_epoch195.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/unet_oracle_vlm_counterfactuals",
        "vlm_pred_file": "/work/wildfirerisk/vlm_cot_predictions_complete.json",
        "oracle": True,
        "model_type": FilmUnet,
        "with_counterfactual": True
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/dino_climate/ckpts/fusion_epoch062.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/dino_climate",
        "climate_cond": True,
        "model_type": DINOv3FiLMTransformerDecoder,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/oracle_sam_no_cot/ckpts/fusion_epoch380.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/oracle_sam_no_cot",
        "vlm_pred_file": "/work/wildfirerisk/vlm_forward_pass_predictions.json",
        "oracle": True,
        "model_type": SamFiLMTransformerDecoder,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/sam_baseline/ckpts/fusion_epoch200.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/sam_baseline",
        "model_type": SamNoFiLMTransformerDecoder,
    },
]


In [ ]:
eval_folders = [
{
    "satellite_image_path": f"{DATA_DIR}/europe_dataset/no_fire_images",
    "raster_path": f"{DATA_DIR}/europe_dataset/no_fire_masks",
    "out_dir": "europe_negatives",
},
{

    'satellite_image_path': f"{DATA_DIR}/large_dataset/satellite_images_full/test",
    'raster_path': f"{DATA_DIR}/large_dataset/normalised_risk_rasters/test",
    'out_dir': 'testset',
}] + [
{
    "satellite_image_path": f"/work/wildfirerisk/europe_dataset/fire_images/{yr-1}",
    "raster_path": f"/work/wildfirerisk/europe_dataset/fire_masks/{yr}",
    "out_dir": f"europe_fires/{yr}"}
    for yr in range(2018, 2026)]
eval_models_and_params = [
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/oracle_dino/ckpts/fusion_epoch440.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/oracle_dino",
        "vlm_pred_file": "/work/wildfirerisk/vlm_cot_predictions.json",
        "oracle": True,
        "model_type": DINOv3FiLMTransformerDecoder,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/oracle_dino_no_cot/ckpts/fusion_epoch459.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/oracle_dino_no_cot",
        "vlm_pred_file": "/work/wildfirerisk/vlm_forward_pass_predictions.json",
        "oracle": True,
        "model_type": DINOv3FiLMTransformerDecoder,
    },
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/dino_baseline/ckpts/fusion_epoch234.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/dino_baseline",
        "model_type": DINOv3NoFiLMTransformerDecoder,
    },
]

In [ ]:
for map in eval_models_and_params:
   for f in eval_folders:
      out_dir = os.path.join(map['out_dir'], f['out_dir'])
      kwargs = f | map
      kwargs['out_dir'] = out_dir
      print(kwargs['model_cpt'], kwargs['satellite_image_path'])
      print(evaluate(**kwargs))

# AlphaEarth

In [ ]:
import os, re, sys, math, json, time, random, glob
project_dir = f'/home/{os.environ["USER"]}/FireScope'
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from custom_datasets.alphaearth import CustomDataset
from config import DATA_DIR
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

from training.train import SSIM, grad_loss, init_distributed
local_rank, world_size, rank = init_distributed()

device = torch.device(f'cuda:{local_rank}' if torch.cuda.is_available() else 'cpu')

ssim = SSIM().to(device)
huber = nn.SmoothL1Loss(beta=1)
mse = nn.MSELoss()
mae = nn.L1Loss()
from typing import Dict, List, Optional
from PIL import Image as PILImage
from models.vlm_generator import QwenVLTransformerRaster
from models.aefilm import AEFiLM
from prompts import raster_generation_prompt as main_prompt
def evaluate(
    model_cpt: str,
    satellite_image_path: str, # AlphaEarth Embeddings
    raster_path: str,
    cond_dir: str,
    out_dir: str,
    device=device,
    ssim=ssim,
    huber=huber,
    mse=mse,
    mae=mae,
    tensor_datatype: bool = True,           # <<< set to False for Qwen VLM path (PIL images)
    model_kwargs: Optional[dict] = None,    # optional: pass custom ctor args here
):
    """
    If model_type is QwenVLTransformerRaster and tensor_datatype=False:
      - dataset returns PIL images
      - we build text prompts and call model(images, prompts)
    Otherwise:
      - dataset returns tensors and we call model(tensor_img, cond=...)
    """

    # ----------------------------------------------------
    # Instantiate model
    # ----------------------------------------------------
    model_kwargs = model_kwargs or {}

    # load checkpoint
    state = torch.load(model_cpt, map_location="cpu")
    model = AEFiLM(cond_dim=60)
    model.load_state_dict(state["model_state"] if "model_state" in state else state)
    model.to(device)
    model.eval()

    # make output dir
    os.makedirs(out_dir, exist_ok=True)

    # ----------------------------------------------------
    # Dataset / Loader
    # ----------------------------------------------------
    # For Qwen VLM path, we want PIL images. For tensor models, we want tensors.
    image_return_mode =  "tensor"
    val_ds = CustomDataset(
        satellite_image_path,
        raster_path,
        cond_dir,
        image_return=image_return_mode,
    )

    val_sampler = DistributedSampler(val_ds, shuffle=False, drop_last=False) if dist.is_initialized() else None
    val_loader = DataLoader(
        val_ds,
        batch_size=32,  # VLM path is heavier
        shuffle=False,
        sampler=val_sampler,
        num_workers=0,
        drop_last=False,
        collate_fn=(collate_vlm if not tensor_datatype else None),  # <<< KEY FIX
    )

    totals = {"recon": 0.0, "ssim": 0.0, "grad": 0.0, "mse": 0.0, "mae": 0.0}
    n = 0

    with torch.no_grad():
        for batch in tqdm(val_loader):
            img, climate, rast, cheat, base = batch

            img = img.to(device, non_blocking=True)
            cond = climate.to(device, non_blocking=True)
            pred = model(img, cond=cond)
            rast = rast.to(device, non_blocking=True)

            l_recon = huber(pred, rast)
            l_mse   = mse(pred, rast)
            l_mae   = mae(pred, rast)

            pred_01 = (pred * 0.5 + 0.5).clamp(0, 1)
            rast_01 = (rast * 0.5 + 0.5).clamp(0, 1)

            l_ssim = 1.0 - ssim(pred_01, rast_01)
            l_grad = grad_loss(pred, rast)

            bs = rast.size(0)
            totals["recon"] += l_recon.item() * bs
            totals["mse"]   += l_mse.item()   * bs
            totals["mae"]   += l_mae.item()   * bs
            totals["ssim"]  += l_ssim.item()  * bs
            totals["grad"]  += l_grad.item()  * bs
            n += bs

            # --- Optional: save predicted rasters (as .npy) ---
            # (kept same behavior you had, with filename fix)
            preds_np = pred_01.detach().cpu().numpy()  # [B,1,H,W]
            for i in range(bs):
                single_pred = preds_np[i, 0] if preds_np.ndim == 4 else preds_np[i]
                # base may be list of names or a single string
                filename = os.path.basename(base[i]) if isinstance(base, (list, tuple)) else os.path.basename(base)
                name = filename[:-4] if filename.lower().endswith(".npy") else filename

                out_path = os.path.join(out_dir, f"{name}.npy")
                np.save(out_path, single_pred)

    # --- Distributed average of ALL metrics ---
    if dist.is_available() and dist.is_initialized():
        t = torch.tensor(
            [totals["recon"], totals["ssim"], totals["grad"], totals["mse"], totals["mae"], float(n)],
            device=device
        )
        dist.all_reduce(t, op=dist.ReduceOp.SUM)
        totals["recon"], totals["ssim"], totals["grad"], totals["mse"], totals["mae"], n = t.tolist()

    return {k: v / max(n, 1) for k, v in totals.items()}


In [ ]:
eval_folders = [
{
    "satellite_image_path": f"{DATA_DIR}/europe_dataset/no_fire_alphaearth_embeddings",
    "raster_path": f"{DATA_DIR}/europe_dataset/no_fire_masks",
    "out_dir": "europe_negatives",
    "cond_dir": '/work/wildfirerisk/europe_dataset/climate_data_3.json'
},
{
    'satellite_image_path': f"{DATA_DIR}/large_dataset/alphaearth_embeddings/test",
    'raster_path': f"{DATA_DIR}/large_dataset/normalised_risk_rasters/test",
    'out_dir': 'testset',
    "cond_dir": '/work/wildfirerisk/large_dataset/climate_data.json'
}] + [
{
    "satellite_image_path": f"/work/wildfirerisk/europe_dataset/fire_alphaearth_embeddings_ada/{yr}",
    "raster_path": f"/work/wildfirerisk/europe_dataset/fire_masks/{yr}",
    "out_dir": f"europe_fires/{yr}",
    "cond_dir": '/work/wildfirerisk/europe_dataset/climate_data.json'}
    for yr in range(2018, 2026)]

eval_models_and_params = [
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/aefilm_climate/ckpts/fusion_epoch975.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/aefilm_climate",
        # "cond_dir": '/work/wildfirerisk/vlm_forward_pass_predictions.json'
    },
]

for map in eval_models_and_params:
   for f in eval_folders:
      out_dir = os.path.join(map['out_dir'], f['out_dir'])
      kwargs = f | map
      kwargs['out_dir'] = out_dir
      print(kwargs['model_cpt'], kwargs['satellite_image_path'])
      print(evaluate(**kwargs))


In [ ]:
eval_folders = [
{
    "satellite_image_path": f"{DATA_DIR}/europe_dataset/no_fire_alphaearth_embeddings",
    "raster_path": f"{DATA_DIR}/europe_dataset/no_fire_masks",
    "out_dir": "europe_negatives",
},
{
    'satellite_image_path': f"{DATA_DIR}/large_dataset/alphaearth_embeddings/test",
    'raster_path': f"{DATA_DIR}/large_dataset/normalised_risk_rasters/test",
    'out_dir': 'testset',
}] + [
{
    "satellite_image_path": f"/work/wildfirerisk/europe_dataset/fire_alphaearth_embeddings_ada/{yr}",
    "raster_path": f"/work/wildfirerisk/europe_dataset/fire_masks/{yr}",
    "out_dir": f"europe_fires/{yr}"}
    for yr in range(2018, 2026)]

eval_models_and_params = [
    {
        "model_cpt": f"{DATA_DIR}/trainings/small/oracle_aefilm_no_cot/ckpts/fusion_epoch975.pt",
        "out_dir": f"{DATA_DIR}/evaluations/small/oracle_aefilm_no_cot",
        "cond_dir": '/work/wildfirerisk/vlm_forward_pass_predictions.json'
    },
]

for map in eval_models_and_params:
   for f in eval_folders:
      out_dir = os.path.join(map['out_dir'], f['out_dir'])
      kwargs = f | map
      kwargs['out_dir'] = out_dir
      print(kwargs['model_cpt'], kwargs['satellite_image_path'])
      print(evaluate(**kwargs))